# Last.fm Tag Normalization

This notebook cleans and standardizes Last.fm genre tags by removing artist names, filtering metadata and mapping synonymous tags to a common representation.

## 1. Load libraries

In [ ]:
# select env music-mood (it handles parquet files well)
import pandas as pd

## 2. Load enriched dataset

Load the dataset produced during the tag enrichment stage.

In [ ]:
model_df = pd.read_parquet("../data/processed/lastfm_scrobbles_with_tags.parquet")

## 3. Initial inspection

Inspect the enriched dataset before tag normalization.

In [ ]:
model_df.shape

In [ ]:
model_df.memory_usage(deep=True).sum() / 1e9

In [ ]:
model_df["tags_clean"][0]

In [ ]:
model_df["artist_clean"].isna().mean()

In [ ]:
model_df["track_clean"].isna().mean()

In [ ]:
model_df["artist_clean"] = model_df["artist_clean"].fillna("")

## 4. Remove artist names from genre tags

Remove artist names that occasionally appear as Last.fm tags to avoid introducing identity information into genre-based clustering.

In [ ]:
def remove_artist_tag(tags, artist_name):
    artist_name = artist_name.lower()
    return [t for t in tags if artist_name not in t]

## 5. Normalize genre labels

Standardize genre tags by applying text normalization and mapping synonymous labels to a common representation.

In [ ]:
def clean_tag(tag):
    tag = tag.lower().strip()
    tag = tag.replace("-", " ")
    return tag

In [ ]:
TAG_MAP = {
    "alternative rock": "rock",
    "indie rock": "indie",
    "indie pop": "indie pop",
    "electronica": "electronic",
    "r&b": "rnb",
    "rhythm and blues": "rnb",
    "synth pop": "synthpop",
    "lo fi": "lo fi",
    "hip hop": "hip hop",
    "pop rock": "rock",
}

In [ ]:
def normalize_tags(tags, artist_name):
    normalized = []
    
    for tag in tags:
        
        if artist_name in tag:
            continue
        
        tag = clean_tag(tag)
        tag = TAG_MAP.get(tag, tag)
        normalized.append(tag)
    
    return list(dict.fromkeys(normalized))  # remove duplicates while preserving order

## 6. Remove metadata and non-genre tags

Filter out tags describing time periods, countries, moods unrelated to genre or other metadata that are not informative for musical clustering.

In [ ]:
REMOVE_METADATA_TAGS = {
    "british", "scottish", "american", "polish", "irish", "usa", "canadian", "english", "french", "german", "swedish", "japanese", "welsh", "manchester",
    "20s", "30s", "40s", "50s", "60s", "70s", "80s", "90s", "00s", "10s",
    "female vocalists", "male vocalists", "singer songwriter",
    "rock", "pop", "electronic"
}

def filter_metadata_tags(tags):
    return [tag for tag in tags if tag not in REMOVE_METADATA_TAGS]

## 7. Apply tag normalization

Apply the complete normalization pipeline to all artist–track combinations.

In [ ]:
model_df["tags_normalized"] = model_df.apply(lambda row: normalize_tags(row["tags_clean"], row["artist_clean"]), axis=1)
model_df["tags_filtered"] = model_df["tags_normalized"].apply(filter_metadata_tags)

## 8. Filtering strategy

Rare-tag filtering was initially considered for count-based vectorization methods. Since TF-IDF naturally downweights infrequent terms, this step was omitted in the final pipeline.

In [ ]:
# from collections import Counter

# all_tags = [t for tags in model_df["tags_filtered"] for t in tags]
# tag_counts = Counter(all_tags)

# MIN_COUNT = 5

# def remove_rare_tags(tags):
#     return [t for t in tags if tag_counts[t] >= MIN_COUNT]

In [ ]:
#model_df["tags_final"] = model_df["tags_filtered"].apply(remove_rare_tags)

## 9. Validation

Inspect normalized tags before exporting the dataset.

In [ ]:
model_df["tags_filtered"][0]

## 10. Export normalized dataset

Save the cleaned and normalized genre-tag dataset for vectorization and clustering.

In [ ]:
model_df.to_parquet("../data/processed/lastfm_scrobbles_clean_tags_final_tfid.parquet", index=False)
model_df.to_csv("../data/processed/lastfm_scrobbles_clean_tags_final_tfid.csv", index=False)

### Output

`data/processed/lastfm_scrobbles_clean_tags_final_tfidf.parquet`

Used in **05_lastfm_tags_clustering.ipynb**.